# AADS-ULoRA Colab: Phase 2 Training (SD-LoRA)

This notebook trains the Phase 2 SD-LoRA (Stable Diffusion LoRA) for data augmentation and domain adaptation.

## What this notebook does:
1. **Load Phase 1 adapter** - Load trained DoRA adapter
2. **Initialize SD-LoRA** - Set up Stable Diffusion model
3. **Train with augmentation** - Generate synthetic data
4. **Monitor training** - Real-time metrics and memory usage
5. **Save checkpoints** - To Google Drive

---
**Expected Runtime:** 1-3 hours (depending on GPU)
**Required GPU:** Any GPU (T4, P100, A100)

---

## 1. Setup and Configuration

In [ ]:
import sys
from pathlib import Path
import json
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def gate_check(step_id: str, check_name: str, condition: bool, expected: str, actual: str):
    status = "PASS" if condition else "FAIL"
    print(f"[{step_id}] {status} :: {check_name} | expected={expected} | actual={actual}")
    if not condition:
        raise RuntimeError(f"Gate failed at {step_id}::{check_name}")

# Add workspace to path
workspace_dir = Path('/content/aads_ulora')
sys.path.insert(0, str(workspace_dir))
gate_check('PHASE2_SETUP', 'workspace_exists', workspace_dir.exists(), 'existing workspace directory', str(workspace_dir))

# Load Colab configuration
config_path = workspace_dir / 'config' / 'colab.json'
gate_check('PHASE2_SETUP', 'config_exists', config_path.exists(), 'colab.json exists', str(config_path))
with open(config_path, 'r') as f:
    config = json.load(f)

gate_check('PHASE2_SETUP', 'phase2_config_present', 'phase2' in config.get('training', {}), 'training.phase2 config present', str(list(config.get('training', {}).keys())))
logger.info(f"✅ Configuration loaded from: {config_path}")
logger.info(f"GPU Type: {config['colab']['gpu_type']}")
logger.info(f"Batch size: {config['training']['phase2']['batch_size']}")

## 2. Import Dependencies

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import time
from tqdm.notebook import tqdm
import psutil
import gc

# Import training module
from src.training.colab_phase2_sd_lora import ColabPhase2Trainer

# Import data utilities
from src.dataset.colab_datasets import ColabCropDataset
from src.dataset.colab_dataloader import ColabDataLoader

# Import Stable Diffusion
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from diffusers import LoraLoaderMixin

gate_check('PHASE2_IMPORTS', 'trainer_data_diffusers_imports', True, 'all imports succeed', 'imports completed')

# Check CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

gate_check('PHASE2_IMPORTS', 'device_ready', True, 'torch initialized', str(torch.device('cuda' if torch.cuda.is_available() else 'cpu')))

## 3. Load Datasets

In [ ]:
# Data paths
data_dir = workspace_dir / 'data'
metadata_path = data_dir / 'dataset_metadata.json'

# Load metadata
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print("Dataset Metadata:")
print(json.dumps(metadata, indent=2))

# Create datasets
from torchvision import transforms

image_size = config['data']['image_size']
normalize_mean = config['data']['normalize_mean']
normalize_std = config['data']['normalize_std']

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std)
])

val_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std)
])

# Assuming data is organized by class folders
train_path = data_dir / 'plantvillage' / 'train'
val_path = data_dir / 'plantvillage' / 'val'

train_dataset = ColabCropDataset(train_path, transform=train_transform)
val_dataset = ColabCropDataset(val_path, transform=val_transform)

print(f"\n✅ Datasets loaded:")
print(f"  Train: {len(train_dataset)} samples, {len(train_dataset.classes)} classes")
print(f"  Val: {len(val_dataset)} samples, {len(val_dataset.classes)} classes")

## 4. Load Phase 1 Adapter

In [ ]:
# Load Phase 1 adapter
phase1_adapter_path = workspace_dir / 'models' / 'phase1_dora_adapter'

if phase1_adapter_path.exists():
    print(f"✅ Phase 1 adapter found at: {phase1_adapter_path}")
else:
    print(f"⚠️  Phase 1 adapter not found. Please run Phase 1 training first.")
    print(f"Expected path: {phase1_adapter_path}")
    raise FileNotFoundError(f"Phase 1 adapter not found")

# Get Phase 2 config
phase2_config = config['training']['phase2']

# Create trainer
trainer = ColabPhase2Trainer(
    adapter_path=str(phase1_adapter_path),
    lora_r=phase2_config['lora_r'],
    lora_alpha=phase2_config['lora_alpha'],
    gradient_accumulation_steps=config['colab']['training']['gradient_accumulation_steps'],
    device='cuda'
)

# Setup optimizer
trainer.setup_optimizer()

print("✅ Trainer initialized")
print(f"  Model: {phase2_config['model_name']}")
print(f"  LoRA rank: {phase2_config['lora_r']}")
print(f"  Learning rate: {phase2_config['learning_rate']}")
print(f"  Batch size: {phase2_config['batch_size']}")
print(f"  Epochs: {phase2_config['num_epochs']}")

## 5. Train Model

In [ ]:
# Create data loaders
batch_size = phase2_config['batch_size']

train_loader = ColabDataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=config['colab']['training']['num_workers'],
    pin_memory=config['colab']['training']['pin_memory'],
    prefetch_factor=config['colab']['training']['prefetch_factor']
)

val_loader = ColabDataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=config['colab']['training']['num_workers'],
    pin_memory=config['colab']['training']['pin_memory']
)

# Train
print("\n" + "="*60)
print("🚀 Starting Phase 2 Training")
print("="*60 + "\n")

history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=phase2_config['num_epochs'],
    save_dir=str(workspace_dir / 'checkpoints' / 'phase2')
)

print("\n" + "="*60)
print("✅ Phase 2 Training Complete!")
print("="*60)

## 6. Plot Training Results

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Accuracy
    axes[1].plot(history['train_accuracy'], label='Train Accuracy')
    axes[1].plot(history['val_accuracy'], label='Val Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    # GPU Memory
    if 'gpu_memory' in history and history['gpu_memory']:
        memory_values = [m['allocated_gb'] for m in history['gpu_memory']]
        axes[2].plot(memory_values)
        axes[2].set_xlabel('Step')
        axes[2].set_ylabel('GPU Memory (GB)')
        axes[2].set_title('GPU Memory Usage')
        axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()

# Plot results
plot_training_history(trainer.history)

print("\nFinal metrics:")
print(f"  Best validation accuracy: {max(trainer.history['val_accuracy']):.4f}")
print(f"  Final training loss: {trainer.history['train_loss'][-1]:.4f}")
print(f"  Final validation loss: {trainer.history['val_loss'][-1]:.4f}")

## 7. Save Final Model

In [ ]:
# Save final adapter
save_path = workspace_dir / 'models' / 'phase2_sd_lora_adapter'
trainer.save_adapter(str(save_path))

print(f"✅ Model saved to: {save_path}")

# Save training history
history_path = workspace_dir / 'logs' / 'phase2_history.json'
history_path.parent.mkdir(parents=True, exist_ok=True)

with open(history_path, 'w') as f:
    json.dump(trainer.history, f, indent=2)

print(f"✅ Training history saved to: {history_path}")

print("\n🎉 Phase 2 complete! Ready for Phase 3.")